# HW5: Neural Networks - EuroSAT Image Classification

Starter code for Homework 5, Part (b). Use a **smaller subset** of the EuroSAT dataset (3-4 classes, ~200 images per class for training) to keep runtimes manageable.

EuroSAT: 64×64 RGB satellite images for land-use classification. 10 classes total.

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import torch.optim as optim

In [ ]:
# Transforms for EuroSAT (64x64 RGB images)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))  # ImageNet stats
])

# Load full EuroSAT dataset (downloads automatically)
full_dataset = torchvision.datasets.EuroSAT(root='./data', download=True, transform=transform)

In [ ]:
# EuroSAT has 10 classes. For a quicker run, use only 3-4 classes and limit to ~200 images per class.
# Classes: AnnualCrop, Forest, HerbaceousVegetation, Highway, Industrial, Pasture, PermanentCrop, Residential, River, SeaLake

# Example: subset to first 4 classes, ~200 images per class for training
# You may adjust which classes and how many images to use.

from torch.utils.data import Subset

# Get class indices (EuroSAT stores samples by class in subfolders)
class_names = full_dataset.classes
print('Available classes:', class_names)

# Build indices for subset - you will need to implement this based on how EuroSAT organizes data
# EuroSAT structure: root/2750/ClassName/image1.jpg, ...
# For a quick start: use a random subset of the full dataset
n_total = len(full_dataset)
n_subset = min(800, n_total)  # ~200 per class for 4 classes
indices = torch.randperm(n_total)[:n_subset]
subset_dataset = Subset(full_dataset, indices)

In [ ]:
# 80/20 train-test split
from torch.utils.data import random_split

train_size = int(0.8 * len(subset_dataset))
test_size = len(subset_dataset) - train_size
train_dataset, test_dataset = random_split(subset_dataset, [train_size, test_size])

trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# Define a simple CNN for EuroSAT (64x64 RGB, 10 classes)
# Input: (batch, 3, 64, 64)

class EuroSATNet(nn.Module):
    def __init__(self, num_classes=10):
        super(EuroSATNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.conv3 = nn.Conv2d(64, 128, 3, 1)
        self.fc1 = nn.Linear(128 * 6 * 6, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv3(x))
        x = F.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = EuroSATNet(num_classes=10)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Training loop - complete as needed for your homework
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print('Using device:', device)

## Your tasks

1. **Subset the data** appropriately (3-4 classes, ~200 images per class for training)
2. **Train the model** and report architecture, training setup, and results
3. **Evaluate** on the test set and summarize accuracy